In [1]:
import numpy as np
import random

# --- PARAMÈTRES ET HYPERPARAMÈTRES (Identiques au précédent) ---
## préparation de structure de notre monde 
GRID_SIZE = 4  # la taille de q-table (4*4)
START_STATE = (0, 0) # la point de départ
GOAL_STATE = (3, 3) # la point d'arrrivée
TRAP_STATE = (1, 1) # le piége
ACTIONS = [0, 1, 2, 3] # 0:Haut, 1:Bas, 2:Gauche, 3:Droite
ACTION_NAMES = ["↑", "↓", "←", "→"]

alpha = 0.1 # le taux d'apprentissage 
gamma = 0.9 # le factuer de réduction ( importance du futur)
epsilon = 0.1 # Exploration ( 10% de chances de choisir au hasard )
episodes = 1000 # la répétition 
q_table = np.zeros((GRID_SIZE, GRID_SIZE, len(ACTIONS))) ## initialisation à zéro

##### --- les régles du monde --##
def get_next_state(state, action):
    row, col = state
    if action == 0 and row > 0: row -= 1 ## déplacement vers le haut
    elif action == 1 and row < 3: row += 1 ## déplacement vers le bas
    elif action == 2 and col > 0: col -= 1 ## déplacement vers la gauche
    elif action == 3 and col < 3: col += 1 ## déplacement vers la droite 
    return (row, col)

##### --- la récompense ( le feedback )--###
def get_reward(state):
    if state == GOAL_STATE: return 10 ## l'agent gagne
    elif state == TRAP_STATE: return -10 ## le piége
    return -1 ## une case vide 

# --- 1. PHASE D'ENTRAÎNEMENT ---
for i in range(episodes):
    state = START_STATE
    while state != GOAL_STATE and state != TRAP_STATE:
        # l'agent tente une nouveauté (hasard), soit il utilise ce qu'il a déjà appris dans sa Q-Table
        # argmax trouve l'indice de la plus grande valeur
        if random.uniform(0, 1) < epsilon:
            action = random.choice(ACTIONS)
        else:
            action = np.argmax(q_table[state[0], state[1]])

        # L'agent exécute l'action, arrive dans la nouvelle case et voit combien de points il a gagné/perdu.
        next_state = get_next_state(state, action)
        reward = get_reward(next_state)

        # utilise équation de bellman pour faire un mise à jour 
        ## equation de bellman : Q[ s , a] = Q[ s , a] + alpha * ( R + (gamma * maxQ[ (s' , a')] - Q[ s , a]))
        old_value = q_table[state[0], state[1], action]
        next_max = np.max(q_table[next_state[0], next_state[1]])
        q_table[state[0], state[1], action] = old_value + alpha * (reward + gamma * next_max - old_value)
        state = next_state

print("--- Entraînement terminé ---\n")

# --- 2. PHASE DE TEST (L'Agent utilise ses connaissances) ---
def test_agent():
    print("Test de l'agent (Chemin optimal) :")
    state = START_STATE
    path = [state]
    total_reward = 0
    steps = 0
    
    # On limite à 20 pas pour éviter les boucles infinies si l'agent est perdu
    while state != GOAL_STATE and steps < 20:
        # Exploitation pure : on prend TOUJOURS la meilleure valeur de la Q-Table
        action = np.argmax(q_table[state[0], state[1]])
        
        state = get_next_state(state, action)
        path.append(state)
        total_reward += get_reward(state)
        steps += 1
        
        if state == TRAP_STATE:
            print("ÉCHEC : L'agent est tombé dans le piège !")
            break
            
    if state == GOAL_STATE:
        print(f"SUCCÈS : Objectif atteint en {steps} étapes !")
    
    # Affichage du chemin sur une grille
    grid = np.full((GRID_SIZE, GRID_SIZE), "·")
    grid[TRAP_STATE] = "X" # Piège
    grid[GOAL_STATE] = "G" # Goal
    for i, s in enumerate(path):
        if grid[s] == "·": grid[s] = str(i)
    
    print(grid)
    print(f"Score total : {total_reward}")

test_agent()

--- Entraînement terminé ---

Test de l'agent (Chemin optimal) :
SUCCÈS : Objectif atteint en 6 étapes !
[['0' '1' '2' '·']
 ['·' 'X' '3' '4']
 ['·' '·' '·' '5']
 ['·' '·' '·' 'G']]
Score total : 5
